In [38]:
import pandas as pd
import numpy as np
import os
# Load data
train_df = pd.read_parquet('../data/raw/UNSW_NB15_training-set.parquet')
test_df = pd.read_parquet('../data/raw/UNSW_NB15_testing-set.parquet')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"\nTrain columns:\n{train_df.columns.tolist()}")
print(f"\nFirst 3 rows:\n{train_df.head(3)}")

Train shape: (175341, 36)
Test shape: (82332, 36)

Train columns:
['dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'is_sm_ips_ports', 'attack_cat', 'label']

First 3 rows:
        dur proto service state  spkts  dpkts  sbytes  dbytes       rate  \
0  0.121478   tcp       -   FIN      6      4     258     172  74.087486   
1  0.649902   tcp       -   FIN     14     38     734   42014  78.473373   
2  1.623129   tcp       -   FIN      8     16     364   13186  14.170161   

          sload  ...  trans_depth  response_body_len  ct_src_dport_ltm  \
0  14158.942383  ...            0                  0                 1   
1   8395.112305  ...            0                  0         

In [17]:
# Check data types and missing values
print("Data Types:")
print(train_df.dtypes.value_counts())
print(f"\nMissing values: {train_df.isnull().sum().sum()}")
print(f"Duplicate rows: {train_df.duplicated().sum()}")

Data Types:
float32     11
int16        9
int8         7
int32        3
int64        2
category     1
category     1
category     1
category     1
Name: count, dtype: int64

Missing values: 0
Duplicate rows: 78519


In [18]:
# Check attack distribution
print("Attack category distribution:")
print(train_df['attack_cat'].value_counts())
print(f"\nLabel distribution (0=Normal, 1=Attack):")
print(train_df['label'].value_counts())

Attack category distribution:
attack_cat
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Analysis           2000
Backdoor           1746
Shellcode          1133
Worms               130
Name: count, dtype: int64

Label distribution (0=Normal, 1=Attack):
label
1    119341
0     56000
Name: count, dtype: int64


In [19]:
# Check categorical columns
cat_cols = ['proto', 'service', 'state']
for col in cat_cols:
    print(f"\n{col} unique values: {train_df[col].nunique()}")
    print(f"Sample values: {train_df[col].unique()[:5]}")


proto unique values: 133
Sample values: ['tcp', 'udp', 'arp', 'ospf', 'icmp']
Categories (133, str): ['3pc', 'a/n', 'aes-sp3-d', 'any', ..., 'xnet', 'xns-idp', 'xtp', 'zero']

service unique values: 13
Sample values: ['-', 'ftp', 'smtp', 'snmp', 'http']
Categories (13, str): ['-', 'dhcp', 'dns', 'ftp', ..., 'smtp', 'snmp', 'ssh', 'ssl']

state unique values: 9
Sample values: ['FIN', 'INT', 'CON', 'ECO', 'REQ']
Categories (9, str): ['CON', 'ECO', 'FIN', 'INT', ..., 'REQ', 'RST', 'URN', 'no']


In [20]:
# Find and display duplicate rows
duplicates = train_df[train_df.duplicated(keep=False)]
duplicates_sorted = duplicates.sort_values(by=list(train_df.columns[:5]))

print(f"Total duplicate rows: {len(duplicates)}")
print(f"\nFirst 5 duplicate rows (showing first 6 columns):")
print(duplicates_sorted.iloc[:5, :6])

Total duplicate rows: 87593

First 5 duplicate rows (showing first 6 columns):
      dur proto service state  spkts  dpkts
57    0.0   arp       -   INT      1      0
58    0.0   arp       -   INT      1      0
938   0.0   arp       -   INT      1      0
939   0.0   arp       -   INT      1      0
3312  0.0   arp       -   INT      1      0


In [21]:
# Find duplicate rows and display them completely
duplicate_mask = train_df.duplicated(keep=False)
duplicate_rows = train_df[duplicate_mask]

print(f"Total rows involved in duplicates: {len(duplicate_rows)}")
print(f"Number of unique duplicate patterns: {duplicate_rows.duplicated().sum()}")

# Show first 2 duplicate groups
duplicate_groups = duplicate_rows[duplicate_rows.duplicated(keep='first')].head(2)

print("\n First duplicate row (showing all columns):")
print(duplicate_groups.iloc[0].to_string())

print("\n Second duplicate row (identical to first):")
print(duplicate_groups.iloc[1].to_string())

# Check if they are truly identical
if len(duplicate_groups) >= 2:
    are_identical = duplicate_groups.iloc[0].equals(duplicate_groups.iloc[1])
    print(f"\n Are the two rows completely identical? {are_identical}")

Total rows involved in duplicates: 87593
Number of unique duplicate patterns: 78519

 First duplicate row (showing all columns):
dur                     0.0
proto                   arp
service                   -
state                   INT
spkts                     1
dpkts                     0
sbytes                   46
dbytes                    0
rate                    0.0
sload                   0.0
dload                   0.0
sloss                     0
dloss                     0
sinpkt                  0.0
dinpkt                  0.0
sjit                    0.0
djit                    0.0
swin                      0
stcpb                     0
dtcpb                     0
dwin                      0
tcprtt                  0.0
synack                  0.0
ackdat                  0.0
smean                    46
dmean                     0
trans_depth               0
response_body_len         0
ct_src_dport_ltm          1
ct_dst_sport_ltm          1
is_ftp_login              0
ct_

In [22]:
# Compare two duplicate rows and find differences
dup1 = duplicate_groups.iloc[0]
dup2 = duplicate_groups.iloc[1]

print("Columns where values differ:")
for col in train_df.columns:
    if dup1[col] != dup2[col]:
        print(f"{col}: {dup1[col]} vs {dup2[col]}")

Columns where values differ:
dur: 0.0 vs 50.004398345947266
proto: arp vs ospf
spkts: 1 vs 6
sbytes: 46 vs 384
rate: 0.0 vs 0.09999100118875504
sload: 0.0 vs 51.19549560546875
sinpkt: 0.0 vs 10000.8798828125
smean: 46 vs 64
ct_dst_sport_ltm: 1 vs 2
is_sm_ips_ports: 1 vs 0


In [23]:
# Keep all data, no dropping
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Separate features and labels
X = train_df.drop(columns=['attack_cat', 'label'])
y = train_df['label']

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Attack ratio: {y.mean()*100:.2f}%")

Train shape: (175341, 36)
Test shape: (82332, 36)
Features shape: (175341, 34)
Labels shape: (175341,)
Attack ratio: 68.06%


In [27]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['proto', 'service', 'state']

for col in cat_cols:
    # Fit on training data only
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    
    # For test, replace unseen values with most frequent value from train
    test_df[col] = test_df[col].astype(str)
    unseen_mask = ~test_df[col].isin(le.classes_)
    if unseen_mask.any():
        most_frequent = train_df[col].mode()[0]
        test_df.loc[unseen_mask, col] = le.inverse_transform([most_frequent])[0]
    
    test_df[col] = le.transform(test_df[col])
    print(f"Encoded {col}")

# Separate features and labels
X_train = train_df.drop(columns=['attack_cat', 'label'])
y_train = train_df['label']

X_test = test_df.drop(columns=['attack_cat', 'label'])
y_test = test_df['label']

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\ny_train distribution:")
print(y_train.value_counts())

Encoded proto
Encoded service
Encoded state

X_train shape: (175341, 34)
X_test shape: (82332, 34)

y_train distribution:
label
1    119341
0     56000
Name: count, dtype: int64


In [29]:
# Find which column contains 'ACC'
for col in ['proto', 'service', 'state']:
    if 'ACC' in test_df[col].astype(str).values:
        print(f"Column '{col}' has value 'ACC'")
        print(f"Unique values in test: {test_df[col].unique()[:10]}")
        print(f"Unique values in train: {train_df[col].unique()[:10]}")

Column 'state' has value 'ACC'
Unique values in test: ['INT', 'FIN', 'REQ', 'ACC', 'CON', 'RST', 'CLO']
Categories (7, str): ['ACC', 'CLO', 'CON', 'FIN', 'INT', 'REQ', 'RST']
Unique values in train: [2 3 0 1 5 6 4 7 8]


In [30]:
# Reload data
train_df = pd.read_parquet('/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/raw/UNSW_NB15_training-set.parquet')
test_df = pd.read_parquet('/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/raw/UNSW_NB15_testing-set.parquet')

# Check which column has 'ACC'
for col in ['proto', 'service', 'state']:
    if 'ACC' in test_df[col].astype(str).values:
        print(f"Column '{col}' has value 'ACC'")
        print(f"Test unique values in {col}: {test_df[col].unique()[:10]}")
        print(f"Train unique values in {col}: {train_df[col].unique()[:10]}")

Column 'state' has value 'ACC'
Test unique values in state: ['INT', 'FIN', 'REQ', 'ACC', 'CON', 'RST', 'CLO']
Categories (7, str): ['ACC', 'CLO', 'CON', 'FIN', 'INT', 'REQ', 'RST']
Train unique values in state: ['FIN', 'INT', 'CON', 'ECO', 'REQ', 'RST', 'PAR', 'URN', 'no']
Categories (9, str): ['CON', 'ECO', 'FIN', 'INT', ..., 'REQ', 'RST', 'URN', 'no']


In [32]:
print("X_train dtypes:")
print(X_train.dtypes.value_counts())

print("\nX_test dtypes:")
print(X_test.dtypes.value_counts())

print("\nSample of X_train:")
print(X_train.head(2))

X_train dtypes:
float32    11
int16       9
int8        6
int64       5
int32       3
Name: count, dtype: int64

X_test dtypes:
float32    11
int16       9
int8        6
int64       5
int32       3
Name: count, dtype: int64

Sample of X_train:
        dur  proto  service  state  spkts  dpkts  sbytes  dbytes       rate  \
0  0.121478     17        0      2      6      4     258     172  74.087486   
1  0.649902     17        0      2     14     38     734   42014  78.473373   

          sload  ...  smean  dmean  trans_depth  response_body_len  \
0  14158.942383  ...     43     43            0                  0   
1   8395.112305  ...     52   1106            0                  0   

   ct_src_dport_ltm  ct_dst_sport_ltm  is_ftp_login  ct_ftp_cmd  \
0                 1                 1             0           0   
1                 1                 1             0           0   

   ct_flw_http_mthd  is_sm_ips_ports  
0                 0                0  
1                 0        

In [33]:
from imblearn.over_sampling import SMOTE
from collections import Counter

print(f"Before SMOTE: {Counter(y_train)}")

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"After SMOTE: {Counter(y_train_balanced)}")
print(f"X_train_balanced shape: {X_train_balanced.shape}")

Before SMOTE: Counter({1: 119341, 0: 56000})
After SMOTE: Counter({0: 119341, 1: 119341})
X_train_balanced shape: (238682, 34)


In [34]:
print("y_train unique values:", y_train.unique())
print("y_train dtype:", y_train.dtype)
print("y_train value counts:")
print(y_train.value_counts())

y_train unique values: [0 1]
y_train dtype: int8
y_train value counts:
label
1    119341
0     56000
Name: count, dtype: int64


In [37]:
#  Save Files
np.save('/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/processed/X_train_balanced.npy', X_train_balanced)
np.save('/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/processed/y_train_balanced.npy', y_train_balanced)

In [41]:
import numpy as np
import os

# Load balanced data
X = np.load('/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/processed/X_train_balanced.npy')
y = np.load('/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/processed/y_train_balanced.npy')

print(f"Total samples: {len(X)}")
print(f"Features: {X.shape[1]}")

# Split into 3 nodes
indices = np.arange(len(X))
np.random.seed(42)
np.random.shuffle(indices)

split_size = len(X) // 3

node1_idx = indices[:split_size]
node2_idx = indices[split_size:2*split_size]
node3_idx = indices[2*split_size:]

# Save to edge_nodes folder
np.save('/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/edge_nodes/node1_X.npy', X[node1_idx])
np.save('/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/edge_nodes/node1_y.npy', y[node1_idx])

np.save('/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/edge_nodes/node2_X.npy', X[node2_idx])
np.save('/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/edge_nodes/node2_y.npy', y[node2_idx])

np.save('/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/edge_nodes/node3_X.npy', X[node3_idx])
np.save('/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/edge_nodes/node3_y.npy', y[node3_idx])

print("Node 1 samples:", len(node1_idx))
print("Node 2 samples:", len(node2_idx))
print("Node 3 samples:", len(node3_idx))

Total samples: 238682
Features: 34
Node 1 samples: 79560
Node 2 samples: 79560
Node 3 samples: 79562


In [42]:
# Check class distribution in each node
for node in [1, 2, 3]:
    y_node = np.load(f'/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/edge_nodes/node{node}_y.npy')
    normal = (y_node == 0).sum()
    attack = (y_node == 1).sum()
    print(f"Node {node}: Normal={normal}, Attack={attack}, Total={len(y_node)}")

Node 1: Normal=39790, Attack=39770, Total=79560
Node 2: Normal=39848, Attack=39712, Total=79560
Node 3: Normal=39703, Attack=39859, Total=79562


In [43]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Load original balanced data
X = np.load('/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/processed/X_train_balanced.npy')
y = np.load('/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/processed/y_train_balanced.npy')

# Normalize
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)

# Save normalized data
np.save('/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/processed/X_train_normalized.npy', X_normalized)

# Resplit normalized data to nodes
indices = np.arange(len(X_normalized))
np.random.seed(42)
np.random.shuffle(indices)

split_size = len(X_normalized) // 3

for i, node in enumerate([1, 2, 3], 1):
    start = (i-1) * split_size
    end = i * split_size if i < 3 else len(X_normalized)
    node_idx = indices[start:end]
    
    np.save(f'/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/edge_nodes/node{node}_X.npy', X_normalized[node_idx])
    np.save(f'/Users/ma-66/Desktop/p1/FedX-ZT-5G/data/edge_nodes/node{node}_y.npy', y[node_idx])
    
    print(f"Node {node}: {len(node_idx)} samples")

print("\nNormalized data saved and redistributed to nodes")

Node 1: 79560 samples
Node 2: 79560 samples
Node 3: 79562 samples

Normalized data saved and redistributed to nodes
